## Note
This notebook is designed to re-format the data to run the orchestrator used in this study. The said study is related to the publication below, and should be mentioned whenever used. Pay attention to the TODO flags, since mis-uses can overwrite some original data.

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from utils.utils import read_list_of_basins
from post_process_v2.pp_utils import get_lstm_predictions, get_sma_predictions, get_all_pet_from_sac, get_flow_and_forcing, rename_camels_for_mlp

## General arguments

In [ ]:
## General arguments V2
CAMELS_CASE = "us"
BASE_PATH_DIR = r"Y:\repo_egu24"

FORCING_SOURCE="maurer"
DATA_DIR = BASE_PATH_DIR+ f"/data_paper/data/camels{CAMELS_CASE}"
CAMELS_ROOT = DATA_DIR + "/camels_us"

FORCING_DIR = CAMELS_ROOT + f"/basin_mean_forcing/{FORCING_SOURCE}"
FLOW_DIR = CAMELS_ROOT + "/usgs_streamflow"

# basins list
BASINS_531 = read_list_of_basins(path_or_list=DATA_DIR + "/basins_list")
BASINS_531.sort()

# SAC-SMA (Newman et al., 2017)
SACSMA_RUN_DIR = CAMELS_ROOT + f"/sacsma_output/{FORCING_SOURCE}"

# LSTM (Kratzert et al., 2019)
RUNS_BASE = BASE_PATH_DIR + "/data_paper/runs"
LSTM_RUNS_DIR = RUNS_BASE + f"/lstm_mse_with_static_{CAMELS_CASE}"

## Benchmark models info

In [ ]:
lstm_runs = get_lstm_predictions(run_dir=LSTM_RUNS_DIR, mse=True)

In [ ]:
sacsma_runs = get_sma_predictions(run_dir=SACSMA_RUN_DIR, basins_list=BASINS_531[:5],save=False)  # TODO: set to True to ease reloading for next runs, and turn ON for all basins
# sacsma_runs = pd.read_parquet("sacsma_outputs.parquet.gzip", engine="pyarrow")  # TODO: uncomment if reloaded

In [ ]:
# Note that usually, we take the median values regarding the initializations. This choice has been mistakenly forgotten on one of the benchmarks. However, the related impact is insignificant 
agg_lstm = lstm_runs.mean(axis=1).to_frame("Q_lstm")
agg_sac= sacsma_runs.median(axis=1).to_frame("Q_SAC_SMA")

### Get ground data

In [ ]:
# PET is from the runs SAC-SMA models output
PET = get_all_pet_from_sac(SACSMA_RUN_DIR, BASINS_531, True) # TODO: comment this and uncomment the second if reloaded
# PET = pd.read_csv("pet_sacsma.txt", sep=";", index_col="Date", parse_dates=["Date"])  

#### Prepare the data for the orchestrator.

Get the forcing alongside the flow, join PET, join Benchmark outputs, compute difference (Obs-Sim), save the file into a folder. 

In [ ]:
path_to = str(Path(CAMELS_ROOT).parent)+"/data/all_sim_obs_lstm"  # TODO: be careful, check this path before uncomment the last line
os.makedirs(path_to, exist_ok=True)
for basin in tqdm(BASINS_531[:5]):  # TODO, adapt to use all basins, get ird of [:], and uncomment the last line
    data_i = get_flow_and_forcing(flow_and_forcing_dir=CAMELS_ROOT, forcing_src="maurer", is_extended=True, basin_id=basin)
    bx_lstm, bx_sac = agg_lstm.xs(basin, level="basin"), agg_sac.xs(basin, level="basin")
    bx_lstm, bx_sac = bx_lstm[~bx_lstm.index.duplicated(keep="first")], bx_sac[~bx_sac.index.duplicated(keep="first")]
    data_b = pd.concat([rename_camels_for_mlp(data_i), PET[basin].to_frame("ETP_CM"), bx_lstm, bx_sac], axis=1, join="inner")
    data_b["Q_err_LSTM"] = data_b.Q_Obs - data_b.Q_lstm
    data_b["Q_err_SACSMA"] = data_b.Q_Obs - data_b.Q_SAC_SMA
    data_b=data_b.astype(np.float32)
    # data_b.to_csv(path_to+f"/{basin}.txt", sep=";", index_label="Date")